### Ingestão de inflação

In [0]:
%sql
select * 

from csv.`/Volumes/workspace/default/raw_data/br_ibge_ipca_mes_brasil.csv`

limit 50;

1. Criar uma tabela de inflacao_row
2. Transformação nos dados -> criando outra tabela 
3. Fazer uma analise de inflação acumulada

## Criar tabela inflacao_row


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.br_ibge_ipca_mes_brasil
USING delta -- tabela final será delta, habilita replace table
AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/raw_data/br_ibge_ipca_mes_brasil.csv',
    format => 'csv',
    header => true,
    inferSchema => true,
    schemaEvolutionMode => 'none' -- desabilita _rescued_data col (schema inference)       
);



In [0]:
%sql
select * from workspace.default.br_ibge_ipca_mes_brasil limit 10;

In [0]:
%sql
ALTER TABLE workspace.default.br_ibge_ipca_mes_brasil RENAME TO workspace.default.inflacao_raw;

### Lendo tabela -> **datafrane**

In [0]:
database_name = 'workspace.default'
table_name = 'inflacao_raw'

df_raw = spark.read.table(f'{database_name}.{table_name}')
display(df_raw.limit(10))

In [0]:
df_raw.show(50)

ajustes de null

In [0]:
from pyspark.sql.functions import col

df_raw.filter(col("indice") == 0.00000).count()

### Por que essa divisão por meses e não só por anos?
### 
A inflação no Brasil era tão agressiva que o governo mudava de moeda no meio do ano civil. Agrupar puramente por "ano" causaria distorções severas na sua análise. Por exemplo:Em Março de 1990, o Plano Collor mudou o Cruzado Novo para Cruzeiro.Em Julho de 1994, o Plano Real entrou em vigor, sepultando o Cruzeiro Real.

In [0]:
from pyspark.sql import functions as F

df_com_moedas = df_raw.withColumn(
    "periodo_moeda",
    F.when(F.col("ano") < 1986, "Cruzeiro (Antigo)")
    .when((F.col("ano") == 1986) | ((F.col("ano") == 1987) & (F.col("mes") <= 2)), "Cruzado")
    .when(((F.col("ano") == 1987) & (F.col("mes") >= 3)) | (F.col("ano") == 1988) | ((F.col("ano") == 1989) & (F.col("mes") <= 1)), "Cruzado (Pós-Reforma)")
    .when(((F.col("ano") == 1989) & (F.col("mes") >= 2)) | ((F.col("ano") == 1990) & (F.col("mes") <= 2)), "Cruzado Novo")
    .when(((F.col("ano") == 1990) & (F.col("mes") >= 3)) | F.col("ano").isin(1991, 1992) | ((F.col("ano") == 1993) & (F.col("mes") <= 7)), "Cruzeiro (Plano Collor)")
    .when(((F.col("ano") == 1993) & (F.col("mes") >= 8)) | ((F.col("ano") == 1994) & (F.col("mes") <= 6)), "Cruzeiro Real")
    .otherwise("Real")
)

display(df_com_moedas)


Criando PK

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

inflacao_cleaned = df_raw 

# 1. Remove linhas duplicadas olhando estritamente para Ano e Mês
df_sem_duplicatas = inflacao_cleaned.dropDuplicates(["ano", "mes"])

# 2. Cria uma janela ordenada por ano e mês de forma crescente
janela_cronologica = Window.orderBy("ano", "mes")

# 3. Gera o ID sequencial baseado na ordem do tempo
df_pk = df_sem_duplicatas.withColumn("id", F.row_number().over(janela_cronologica))

# 4. Adiciona a coluna do período da moeda
df_final = df_pk.withColumn(
    "periodo_moeda",
    F.when(F.col("ano") < 1986, "Cruzeiro (Antigo)")
    .when((F.col("ano") == 1986) | ((F.col("ano") == 1987) & (F.col("mes") <= 2)), "Cruzado")
    .when(((F.col("ano") == 1987) & (F.col("mes") >= 3)) | (F.col("ano") == 1988) | ((F.col("ano") == 1989) & (F.col("mes") <= 1)), "Cruzado (Pós-Reforma)")
    .when(((F.col("ano") == 1989) & (F.col("mes") >= 2)) | ((F.col("ano") == 1990) & (F.col("mes") <= 2)), "Cruzado Novo")
    .when(((F.col("ano") == 1990) & (F.col("mes") >= 3)) | F.col("ano").isin(1991, 1992) | ((F.col("ano") == 1993) & (F.col("mes") <= 7)), "Cruzeiro (Plano Collor)")
    .when(((F.col("ano") == 1993) & (F.col("mes") >= 8)) | ((F.col("ano") == 1994) & (F.col("mes") <= 6)), "Cruzeiro Real")
    .otherwise("Real")
)

# Move a coluna 'id' para ser a primeira da tabela
colunas = ["id"] + [col for col in df_final.columns if col != "id"]
df_final = df_final.select(colunas)

display(df_final)


### Escrevendo tabela

In [0]:
path_table = 'workspace.default'
table_name = 'inflacao_cleaned'

df_final.write.mode("overwrite").saveAsTable(f'{path_table}.{table_name}')


In [0]:
df_result = spark.read.table(f'{path_table}.{table_name}')
display(df_result.limit(10))

### O que evitar

In [0]:
.toPandas() -> pandas.DataFrame -> Sem aproveitamento do paralelismo
.collect() -> lista -> Sem aproveitamento do paralelismo